In [2]:
# ============================================================
# run_weekly.py — AI Stock Screener (Indian Markets)
# Day 10: Weekly Automation
# Run every weekend: python run_weekly.py
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
import pickle
import os
import time
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# ── PATHS ────────────────────────────────────────────────────
BASE_DIR    = r'E:\learn\Project 1 AI Screener\stock-ai-india'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
MODELS_DIR  = os.path.join(BASE_DIR, 'models')
REPORTS_DIR = os.path.join(DATA_DIR, 'weekly_reports')

os.makedirs(REPORTS_DIR, exist_ok=True)

print("=" * 60)
print("  AI Stock Screener — Weekly Run")
print(f"  Date : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 60)
print(f"  Data dir    : {DATA_DIR}")
print(f"  Models dir  : {MODELS_DIR}")
print(f"  Reports dir : {REPORTS_DIR}")
print("✅ Cell 1 — Imports & Setup done")

  AI Stock Screener — Weekly Run
  Date : 2026-04-12 20:54
  Data dir    : E:\learn\Project 1 AI Screener\stock-ai-india\data
  Models dir  : E:\learn\Project 1 AI Screener\stock-ai-india\models
  Reports dir : E:\learn\Project 1 AI Screener\stock-ai-india\data\weekly_reports
✅ Cell 1 — Imports & Setup done


In [3]:
# ── CELL 2: LOAD STOCKS + BUILD TICKER MAP ───────────────────

quality_df = pd.read_csv(os.path.join(DATA_DIR, 'quality_passed.csv'))
prefilt_df = pd.read_csv(os.path.join(DATA_DIR, 'prefilt_passed.csv'))

# Build ticker map: Symbol → yfinance ticker (NSE or BSE)
ticker_map = {}
for _, row in prefilt_df.iterrows():
    sym = row['Symbol']
    if row['Exchange'] == 'NSE':
        ticker_map[sym] = f"{sym}.NS"
    else:
        ticker_map[sym] = f"{sym}.BO"

symbols = quality_df['Symbol'].tolist()

print(f"Total stocks to process : {len(symbols)}")
print(f"Ticker map size         : {len(ticker_map)}")
print(f"\nSample tickers:")
for sym in symbols[:5]:
    print(f"  {sym} → {ticker_map.get(sym, sym + '.NS')}")

print("\n✅ Cell 2 — Stocks loaded")

Total stocks to process : 752
Ticker map size         : 2142

Sample tickers:
  20MICRONS → 20MICRONS.NS
  360ONE → 360ONE.NS
  3MINDIA → 3MINDIA.NS
  5PAISA → 5PAISA.NS
  AADHARHFC → AADHARHFC.NS

✅ Cell 2 — Stocks loaded


In [4]:
# ── CELL 3: INCREMENTAL PRICE UPDATE ─────────────────────────
# For each stock, downloads only from last available date → today
# First run after a gap: fetches exactly the missing days
# Future weekly runs: fetches only ~5-7 days

PRICE_FILE = os.path.join(DATA_DIR, 'price_data_full.pkl')

# Load existing price data
with open(PRICE_FILE, 'rb') as f:
    price_data = pickle.load(f)

print(f"Loaded existing price data : {len(price_data)} stocks")
print(f"Started : {datetime.now().strftime('%H:%M:%S')}")
print("-" * 60)

updated = 0
skipped = 0
failed  = []

for i, symbol in enumerate(symbols):
    ticker = ticker_map.get(symbol, f"{symbol}.NS")

    try:
        existing_df = price_data.get(symbol)

        # Find start date for incremental fetch
        if existing_df is not None and len(existing_df) > 0:
            last_date  = existing_df.index[-1]
            # Convert to tz-naive if needed
            if hasattr(last_date, 'tzinfo') and last_date.tzinfo is not None:
                last_date = last_date.tz_localize(None)
            start_date = (last_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        else:
            # Stock missing entirely — full download
            start_date = '2010-01-01'

        today = datetime.now().strftime('%Y-%m-%d')

        # Skip if already up to date
        if start_date >= today:
            skipped += 1
            continue

        # Download only the missing window
        new_df = yf.download(ticker, start=start_date, end=today,
                             interval='1d', progress=False,
                             auto_adjust=True)

        # Flatten MultiIndex
        if isinstance(new_df.columns, pd.MultiIndex):
            new_df.columns = [col[0] for col in new_df.columns]

        new_df = new_df.loc[:, ~new_df.columns.duplicated()]

        if new_df is not None and len(new_df) > 0:
            if existing_df is not None:
                # Normalize index timezone before concat
                if hasattr(existing_df.index, 'tz') and existing_df.index.tz is not None:
                    existing_df.index = existing_df.index.tz_localize(None)
                if hasattr(new_df.index, 'tz') and new_df.index.tz is not None:
                    new_df.index = new_df.index.tz_localize(None)

                combined = pd.concat([existing_df, new_df])
                combined = combined[~combined.index.duplicated(keep='last')]
                combined = combined.sort_index()
                price_data[symbol] = combined
            else:
                price_data[symbol] = new_df

            updated += 1
        else:
            skipped += 1

    except Exception as e:
        failed.append((symbol, str(e)[:60]))

    # Progress every 50
    if (i + 1) % 50 == 0 or (i + 1) == len(symbols):
        pct = (i + 1) / len(symbols) * 100
        print(f"  [{i+1:4d}/{len(symbols)}] {pct:5.1f}% | "
              f"Updated: {updated} | Skipped: {skipped} | "
              f"Failed: {len(failed)} | "
              f"Time: {datetime.now().strftime('%H:%M:%S')}")

    time.sleep(0.5)

# Save updated pkl
with open(PRICE_FILE, 'wb') as f:
    pickle.dump(price_data, f)

print(f"\n✅ Cell 3 — Incremental price update complete")
print(f"   Updated : {updated}")
print(f"   Skipped : {skipped} (already current)")
print(f"   Failed  : {len(failed)}")
if failed:
    print(f"\n   Failed symbols (first 10):")
    for sym, err in failed[:10]:
        print(f"     {sym}: {err}")

Loaded existing price data : 752 stocks
Started : 20:59:02
------------------------------------------------------------
  [  50/752]   6.6% | Updated: 50 | Skipped: 0 | Failed: 0 | Time: 20:59:49
  [ 100/752]  13.3% | Updated: 100 | Skipped: 0 | Failed: 0 | Time: 21:00:53



1 Failed download:
['COMSYN.NS']: TypeError("'NoneType' object is not subscriptable")


  [ 150/752]  19.9% | Updated: 149 | Skipped: 1 | Failed: 0 | Time: 21:01:55
  [ 200/752]  26.6% | Updated: 199 | Skipped: 1 | Failed: 0 | Time: 21:03:07
  [ 250/752]  33.2% | Updated: 249 | Skipped: 1 | Failed: 0 | Time: 21:04:08
  [ 300/752]  39.9% | Updated: 299 | Skipped: 1 | Failed: 0 | Time: 21:05:25



1 Failed download:
['INNOVACAP.NS']: TypeError("'NoneType' object is not subscriptable")


  [ 350/752]  46.5% | Updated: 348 | Skipped: 2 | Failed: 0 | Time: 21:06:43
  [ 400/752]  53.2% | Updated: 398 | Skipped: 2 | Failed: 0 | Time: 21:07:48
  [ 450/752]  59.8% | Updated: 448 | Skipped: 2 | Failed: 0 | Time: 21:08:38
  [ 500/752]  66.5% | Updated: 498 | Skipped: 2 | Failed: 0 | Time: 21:09:31
  [ 550/752]  73.1% | Updated: 548 | Skipped: 2 | Failed: 0 | Time: 21:10:28
  [ 600/752]  79.8% | Updated: 598 | Skipped: 2 | Failed: 0 | Time: 21:11:20
  [ 650/752]  86.4% | Updated: 648 | Skipped: 2 | Failed: 0 | Time: 21:12:07
  [ 700/752]  93.1% | Updated: 698 | Skipped: 2 | Failed: 0 | Time: 21:13:28
  [ 750/752]  99.7% | Updated: 748 | Skipped: 2 | Failed: 0 | Time: 21:14:29
  [ 752/752] 100.0% | Updated: 750 | Skipped: 2 | Failed: 0 | Time: 21:14:31

✅ Cell 3 — Incremental price update complete
   Updated : 750
   Skipped : 2 (already current)
   Failed  : 0


In [5]:
# ── CELL 4: RECOMPUTE TECHNICAL INDICATORS ────────────────────

def flatten_df(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    return df

def compute_indicators(df):
    df = df.copy()

    # ── EMA ───────────────────────────────────────────────────
    df['EMA20']  = df['Close'].ewm(span=20,  adjust=False).mean()
    df['EMA50']  = df['Close'].ewm(span=50,  adjust=False).mean()
    df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean()

    # ── RSI ───────────────────────────────────────────────────
    delta    = df['Close'].diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))

    # ── MACD ──────────────────────────────────────────────────
    ema12          = df['Close'].ewm(span=12, adjust=False).mean()
    ema26          = df['Close'].ewm(span=26, adjust=False).mean()
    macd_line      = ema12 - ema26
    signal_line    = macd_line.ewm(span=9, adjust=False).mean()
    df['MACD Hist'] = macd_line - signal_line

    # ── ADX + DI+ / DI- ──────────────────────────────────────
    high  = df['High']
    low   = df['Low']
    close = df['Close']

    plus_dm  = high.diff()
    minus_dm = -low.diff()
    plus_dm  = plus_dm.where((plus_dm > minus_dm) & (plus_dm > 0), 0)
    minus_dm = minus_dm.where((minus_dm > plus_dm) & (minus_dm > 0), 0)

    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low  - close.shift()).abs()
    ], axis=1).max(axis=1)

    atr14      = tr.ewm(com=13,      adjust=False).mean()
    plus_di14  = 100 * plus_dm.ewm(com=13,  adjust=False).mean() / atr14.replace(0, np.nan)
    minus_di14 = 100 * minus_dm.ewm(com=13, adjust=False).mean() / atr14.replace(0, np.nan)
    dx         = 100 * (plus_di14 - minus_di14).abs() / (plus_di14 + minus_di14).replace(0, np.nan)

    df['ADX']      = dx.ewm(com=13, adjust=False).mean()
    df['DI_Plus']  = plus_di14
    df['DI_Minus'] = minus_di14

    # ── ATR ───────────────────────────────────────────────────
    df['ATR'] = atr14

    # ── VOLUME RATIO ──────────────────────────────────────────
    df['Vol_MA20'] = df['Volume'].rolling(20).mean()
    df['Vol Ratio'] = df['Volume'] / df['Vol_MA20'].replace(0, np.nan)

    return df

# ── RUN FOR ALL STOCKS ────────────────────────────────────────
INDICATOR_FILE = os.path.join(DATA_DIR, 'indicator_data_full.pkl')

print(f"Computing indicators for {len(price_data)} stocks...")
print(f"Started : {datetime.now().strftime('%H:%M:%S')}")
print("-" * 60)

indicator_data = {}
failed         = []

for i, (symbol, df) in enumerate(price_data.items()):
    try:
        ind_df = compute_indicators(df)
        indicator_data[symbol] = ind_df
    except Exception as e:
        failed.append((symbol, str(e)[:60]))

    if (i + 1) % 100 == 0 or (i + 1) == len(price_data):
        pct = (i + 1) / len(price_data) * 100
        print(f"  [{i+1:4d}/{len(price_data)}] {pct:5.1f}% | "
              f"Done: {len(indicator_data)} | Failed: {len(failed)}")

with open(INDICATOR_FILE, 'wb') as f:
    pickle.dump(indicator_data, f)

# Quick sanity check
sample_sym = list(indicator_data.keys())[0]
sample_df  = indicator_data[sample_sym]
print(f"\nSample — {sample_sym} (last row):")
cols = ['Close', 'EMA20', 'EMA50', 'EMA200', 'RSI', 'ADX', 'DI_Plus', 'DI_Minus', 'MACD Hist', 'Vol Ratio']
print(sample_df[cols].iloc[-1].round(2).to_string())

print(f"\n✅ Cell 4 — Indicators computed")
print(f"   Success : {len(indicator_data)}")
print(f"   Failed  : {len(failed)}")
if failed:
    for sym, err in failed[:10]:
        print(f"     {sym}: {err}")

Computing indicators for 752 stocks...
Started : 21:15:24
------------------------------------------------------------
  [ 100/752]  13.3% | Done: 100 | Failed: 0
  [ 200/752]  26.6% | Done: 200 | Failed: 0
  [ 300/752]  39.9% | Done: 300 | Failed: 0
  [ 400/752]  53.2% | Done: 400 | Failed: 0
  [ 500/752]  66.5% | Done: 500 | Failed: 0
  [ 600/752]  79.8% | Done: 600 | Failed: 0
  [ 700/752]  93.1% | Done: 700 | Failed: 0
  [ 752/752] 100.0% | Done: 752 | Failed: 0

Sample — 20MICRONS (last row):
Close        162.37
EMA20        155.93
EMA50        165.52
EMA200       192.88
RSI           53.80
ADX           23.18
DI_Plus       23.37
DI_Minus      23.93
MACD Hist      2.73
Vol Ratio      0.74

✅ Cell 4 — Indicators computed
   Success : 752
   Failed  : 0


In [6]:
# ── CELL 5: BUILD FEATURES ────────────────────────────────────

def build_features(symbol, df):
    df = df.copy()

    # ── RETURNS ───────────────────────────────────────────────
    df['return_1d']  = df['Close'].pct_change(1)
    df['return_5d']  = df['Close'].pct_change(5)
    df['return_20d'] = df['Close'].pct_change(20)
    df['return_60d'] = df['Close'].pct_change(60)

    # ── 52W HIGH / LOW DISTANCE ───────────────────────────────
    df['52w_high']      = df['Close'].rolling(252).max()
    df['52w_low']       = df['Close'].rolling(252).min()
    df['dist_52w_high'] = (df['Close'] - df['52w_high']) / df['52w_high']
    df['dist_52w_low']  = (df['Close'] - df['52w_low'])  / df['52w_low']

    # ── ATR % ─────────────────────────────────────────────────
    df['atr_pct'] = df['ATR'] / df['Close']

    # ── VOLATILITY ────────────────────────────────────────────
    df['volatility_20d'] = df['return_1d'].rolling(20).std()
    df['volatility_60d'] = df['return_1d'].rolling(60).std()

    # ── VOLUME FEATURES ───────────────────────────────────────
    vol_ma5          = df['Volume'].rolling(5).mean()
    vol_ma20         = df['Volume'].rolling(20).mean()
    df['vol_ratio_5d']  = df['Volume'] / vol_ma5.replace(0, np.nan)
    df['vol_ratio_20d'] = df['Volume'] / vol_ma20.replace(0, np.nan)
    df['vol_spike']     = (df['vol_ratio_5d'] > 2.0).astype(int)

    # OBV slope
    obv = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    df['obv_slope_10d'] = obv.diff(10) / (df['Volume'].rolling(10).mean().replace(0, np.nan))

    # ── RSI FEATURES ──────────────────────────────────────────
    df['rsi']           = df['RSI']
    df['rsi_slope_5d']  = df['RSI'].diff(5)
    df['rsi_oversold']  = (df['RSI'] < 30).astype(int)
    df['rsi_overbought']= (df['RSI'] > 70).astype(int)

    # ── MACD FEATURES ─────────────────────────────────────────
    df['macd_hist']    = df['MACD Hist']
    df['macd_slope_3d']= df['MACD Hist'].diff(3)
    df['macd_slope_5d']= df['MACD Hist'].diff(5)
    df['macd_cross']   = (
        (df['MACD Hist'] > 0) & (df['MACD Hist'].shift(1) <= 0)
    ).astype(int)

    # ── ADX FEATURES ──────────────────────────────────────────
    df['adx']       = df['ADX']
    df['adx_slope'] = df['ADX'].diff(5)
    df['di_spread'] = df['DI_Plus'] - df['DI_Minus']

    # ── EMA FEATURES ──────────────────────────────────────────
    df['price_vs_ema20']  = (df['Close'] - df['EMA20'])  / df['EMA20']
    df['price_vs_ema50']  = (df['Close'] - df['EMA50'])  / df['EMA50']
    df['price_vs_ema200'] = (df['Close'] - df['EMA200']) / df['EMA200']
    df['ema20_vs_ema50']  = (df['EMA20'] - df['EMA50'])  / df['EMA50']
    df['ema50_vs_ema200'] = (df['EMA50'] - df['EMA200']) / df['EMA200']

    return df

# ── FEATURE COLUMNS (same 30 as training) ────────────────────
FEATURE_COLS = [
    'return_1d', 'return_5d', 'return_20d', 'return_60d',
    'dist_52w_high', 'dist_52w_low', 'atr_pct',
    'volatility_20d', 'volatility_60d',
    'vol_ratio_5d', 'vol_ratio_20d', 'obv_slope_10d', 'vol_spike',
    'rsi', 'rsi_slope_5d', 'rsi_oversold', 'rsi_overbought',
    'macd_hist', 'macd_slope_3d', 'macd_slope_5d', 'macd_cross',
    'adx', 'adx_slope', 'di_spread',
    'price_vs_ema20', 'price_vs_ema50', 'price_vs_ema200',
    'ema20_vs_ema50', 'ema50_vs_ema200',
]

# ── RUN FOR ALL STOCKS ────────────────────────────────────────
print(f"Building features for {len(indicator_data)} stocks...")

all_features = {}
failed       = []

for symbol, df in indicator_data.items():
    try:
        all_features[symbol] = build_features(symbol, df)
    except Exception as e:
        failed.append((symbol, str(e)[:60]))

# Sanity check
sample_sym = list(all_features.keys())[0]
sample_row = all_features[sample_sym].iloc[-1]

print(f"\nSample — {sample_sym} (last row feature check):")
for col in FEATURE_COLS:
    val = sample_row.get(col, 'MISSING')
    flag = '❌ MISSING' if val == 'MISSING' else ''
    print(f"  {col:25} {round(float(val), 4) if val != 'MISSING' else val} {flag}")

print(f"\n✅ Cell 5 — Features built")
print(f"   Success : {len(all_features)}")
print(f"   Failed  : {len(failed)}")
if failed:
    for sym, err in failed[:10]:
        print(f"     {sym}: {err}")

Building features for 752 stocks...

Sample — 20MICRONS (last row feature check):
  return_1d                 0.0188 
  return_5d                 0.0832 
  return_20d                -0.0273 
  return_60d                -0.1704 
  dist_52w_high             -0.4134 
  dist_52w_low              0.2195 
  atr_pct                   0.0496 
  volatility_20d            0.0417 
  volatility_60d            0.0311 
  vol_ratio_5d              1.0444 
  vol_ratio_20d             0.738 
  obv_slope_10d             0.6982 
  vol_spike                 0.0 
  rsi                       53.8014 
  rsi_slope_5d              10.0456 
  rsi_oversold              0.0 
  rsi_overbought            0.0 
  macd_hist                 2.7271 
  macd_slope_3d             1.4421 
  macd_slope_5d             2.8289 
  macd_cross                0.0 
  adx                       23.1813 
  adx_slope                 -7.0812 
  di_spread                 -0.5593 
  price_vs_ema20            0.0413 
  price_vs_ema50       

In [7]:
# ── CELL 6: LOAD ML MODELS ────────────────────────────────────

print("Loading ML models...")
print("-" * 60)

def load_pkl(filename):
    path = os.path.join(MODELS_DIR, filename)
    with open(path, 'rb') as f:
        return pickle.load(f)

bottom_models        = load_pkl('bottom_models.pkl')
bottom_encoders      = load_pkl('bottom_encoders.pkl')
top_models           = load_pkl('top_models.pkl')
top_encoders         = load_pkl('top_encoders.pkl')
trend_models         = load_pkl('trend_models.pkl')
trend_encoders       = load_pkl('trend_encoders.pkl')
trend_label_encoders = load_pkl('trend_label_encoders.pkl')
forecast_models      = load_pkl('forecast_models.pkl')
forecast_encoders    = load_pkl('forecast_encoders.pkl')
bullish_cont_models  = load_pkl('bullish_cont_models.pkl')
bullish_cont_encoders= load_pkl('bullish_cont_encoders.pkl')
bearish_cont_models  = load_pkl('bearish_cont_models.pkl')
bearish_cont_encoders= load_pkl('bearish_cont_encoders.pkl')
symbol_group         = load_pkl('symbol_group.pkl')
group_stocks         = load_pkl('group_stocks.pkl')

print(f"  bottom_models        : {list(bottom_models.keys())}")
print(f"  top_models           : {list(top_models.keys())}")
print(f"  trend_models         : {list(trend_models.keys())}")
print(f"  forecast_models      : {list(forecast_models.keys())}")
print(f"  bullish_cont_models  : {list(bullish_cont_models.keys())}")
print(f"  bearish_cont_models  : {list(bearish_cont_models.keys())}")
print(f"  symbol_group entries : {len(symbol_group)}")
print(f"  group_stocks entries : {len(group_stocks)}")

print(f"\n✅ Cell 6 — Models loaded")

Loading ML models...
------------------------------------------------------------
  bottom_models        : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  top_models           : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  trend_models         : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  forecast_models      : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  bullish_cont_models  : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  bearish_cont_models  : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
  symbol_group entries : 752
  group_stocks entries : 6

✅ Cell 6 — Models loaded


In [8]:
# ── CELL 7: INFERENCE ENGINE ──────────────────────────────────

def get_latest_features(symbol):
    df = all_features[symbol].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURE_COLS)
    if len(df) == 0:
        return None
    return df.iloc[[-1]]

def get_symbol_enc(symbol, encoder):
    try:
        return encoder.transform([symbol])[0]
    except:
        return 0

def assign_ml_label(best_setup, forecast_25d):
    if best_setup == 'Momentum':
        return 'Bullish Continual' if forecast_25d >= 0 else 'Bearish Continual'
    elif best_setup == 'Reversal':
        return 'Reversal' if forecast_25d >= 0 else 'Bearish'
    elif best_setup == 'Watching':
        return 'Bullish' if forecast_25d >= 0 else 'No Signal'
    else:
        return 'No Signal'

def assign_ml_confidence(ml_label, result):
    if ml_label == 'Bullish Continual':
        return result.get('Bullish_Cont_Prob')
    elif ml_label == 'Bearish Continual':
        return result.get('Bearish_Cont_Prob')
    elif ml_label == 'Reversal':
        return result.get('Bottom_Rev_Prob')
    elif ml_label == 'Bearish':
        return result.get('Top_Rev_Prob')
    elif ml_label == 'Bullish':
        return result.get('Bottom_Rev_Prob')
    else:
        return None

def run_inference(symbol, best_setup):
    result = {
        'Symbol'             : symbol,
        'Group'              : None,
        'Bottom_Rev_Prob'    : None,
        'Top_Rev_Prob'       : None,
        'Bottom_Rev_Flag'    : False,
        'Top_Rev_Flag'       : False,
        'Bullish_Cont_Prob'  : None,
        'Bearish_Cont_Prob'  : None,
        'Forecast_25d_Pct'   : None,
        'Forecast_45d_Pct'   : None,
        'Forecast_180d_Pct'  : None,
        'Forecast_25d_Price' : None,
        'Forecast_45d_Price' : None,
        'Forecast_180d_Price': None,
        'ML_Prediction'      : 'No Signal',
        'ML_Confidence'      : None,
    }

    group = symbol_group.get(symbol)
    if group is None:
        return result
    result['Group'] = group

    latest = get_latest_features(symbol)
    if latest is None:
        return result

    feat_cols = FEATURE_COLS + ['symbol_enc']

    # ── MODEL 1: REVERSAL ─────────────────────────────────────
    if group in bottom_models:
        sym_enc              = get_symbol_enc(symbol, bottom_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        bottom_prob = bottom_models[group].predict_proba(X)[0][1]
        result['Bottom_Rev_Prob'] = round(float(bottom_prob) * 100, 1)
        result['Bottom_Rev_Flag'] = bottom_prob >= 0.60

        top_prob = top_models[group].predict_proba(X)[0][1]
        result['Top_Rev_Prob'] = round(float(top_prob) * 100, 1)
        result['Top_Rev_Flag'] = top_prob >= 0.60

    # ── MODEL 4: CONTINUATION ─────────────────────────────────
    if group in bullish_cont_models:
        sym_enc              = get_symbol_enc(symbol, bullish_cont_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        bc_prob = bullish_cont_models[group].predict_proba(X)[0][1]
        result['Bullish_Cont_Prob'] = round(float(bc_prob) * 100, 1)

    if group in bearish_cont_models:
        sym_enc              = get_symbol_enc(symbol, bearish_cont_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        dc_prob = bearish_cont_models[group].predict_proba(X)[0][1]
        result['Bearish_Cont_Prob'] = round(float(dc_prob) * 100, 1)

    # ── MODEL 3: FORECAST ─────────────────────────────────────
    if group in forecast_models:
        sym_enc              = get_symbol_enc(symbol, forecast_encoders[group]['25d'])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        current_price = all_features[symbol]['Close'].iloc[-1]

        for horizon in ['25d', '45d', '180d']:
            if horizon not in forecast_models[group]:
                continue
            pred_return = forecast_models[group][horizon].predict(X)[0]
            pred_price  = current_price * (1 + pred_return)
            result[f'Forecast_{horizon}_Pct']   = round(float(pred_return) * 100, 1)
            result[f'Forecast_{horizon}_Price']  = round(float(pred_price), 2)

    # ── ASSIGN ML LABEL + CONFIDENCE ──────────────────────────
    forecast_25d          = result['Forecast_25d_Pct'] or 0
    ml_label              = assign_ml_label(best_setup, forecast_25d)
    ml_conf               = assign_ml_confidence(ml_label, result)
    result['ML_Prediction'] = ml_label
    result['ML_Confidence'] = ml_conf

    return result

print("✅ Cell 7 — Inference engine defined")

✅ Cell 7 — Inference engine defined


In [9]:
# ── CELL 8: RUN INFERENCE ON ALL STOCKS ──────────────────────

# Load current technical report to get Best Setup for each stock
tech_df = pd.read_csv(os.path.join(DATA_DIR, 'technical_report_full.csv'))

print(f"Running inference on {len(tech_df)} stocks...")
print(f"Started : {datetime.now().strftime('%H:%M:%S')}")
print("-" * 60)

all_ml_scores = []
failed        = []

for i, row in tech_df.iterrows():
    symbol     = row['Symbol']
    best_setup = row['Best Setup']

    try:
        result = run_inference(symbol, best_setup)
        all_ml_scores.append(result)
    except Exception as e:
        failed.append((symbol, str(e)[:60]))

    if (i + 1) % 100 == 0 or (i + 1) == len(tech_df):
        pct = (i + 1) / len(tech_df) * 100
        print(f"  [{i+1:4d}/{len(tech_df)}] {pct:5.1f}% | "
              f"Done: {len(all_ml_scores)} | Failed: {len(failed)}")

ml_scores_df = pd.DataFrame(all_ml_scores)

print(f"\nML Prediction Distribution:")
print(ml_scores_df['ML_Prediction'].value_counts().to_string())

print(f"\nConfidence stats (buy-side only):")
buy_labels = ['Bullish Continual', 'Bullish', 'Reversal']
for label in buy_labels:
    subset = ml_scores_df[
        (ml_scores_df['ML_Prediction'] == label) &
        (ml_scores_df['ML_Confidence'].notna())
    ]
    if len(subset) > 0:
        print(f"  {label:20} count={len(subset):3d} | "
              f"avg_conf={subset['ML_Confidence'].mean():.1f}% | "
              f"max_conf={subset['ML_Confidence'].max():.1f}%")

print(f"\n✅ Cell 8 — Inference complete")
print(f"   Success : {len(all_ml_scores)}")
print(f"   Failed  : {len(failed)}")
if failed:
    for sym, err in failed[:10]:
        print(f"     {sym}: {err}")

Running inference on 752 stocks...
Started : 21:20:42
------------------------------------------------------------
  [ 100/752]  13.3% | Done: 100 | Failed: 0
  [ 200/752]  26.6% | Done: 200 | Failed: 0
  [ 300/752]  39.9% | Done: 300 | Failed: 0
  [ 400/752]  53.2% | Done: 400 | Failed: 0
  [ 500/752]  66.5% | Done: 500 | Failed: 0
  [ 600/752]  79.8% | Done: 600 | Failed: 0
  [ 700/752]  93.1% | Done: 700 | Failed: 0
  [ 752/752] 100.0% | Done: 752 | Failed: 0

ML Prediction Distribution:
ML_Prediction
Bullish              297
Reversal             285
Bullish Continual    110
Bearish               31
No Signal             21
Bearish Continual      8

Confidence stats (buy-side only):
  Bullish Continual    count=110 | avg_conf=41.6% | max_conf=81.9%
  Bullish              count=297 | avg_conf=10.2% | max_conf=91.0%
  Reversal             count=285 | avg_conf=8.2% | max_conf=88.7%

✅ Cell 8 — Inference complete
   Success : 752
   Failed  : 0


In [19]:
# ── CELL 9: TECHNICAL ANALYSIS — FULL UNIVERSE ───────────────
# Direct automation of Day 8 Cell 12
# Includes: Volume Profile, full momentum/reversal scoring,
#           dynamic consolidation detection, correct tier logic

print("Running full technical analysis...")
print(f"Started : {datetime.now().strftime('%H:%M:%S')}")
print("-" * 60)

TECH_CHECKPOINT = os.path.join(DATA_DIR, 'tech_checkpoint.pkl')

# ── LOAD SUPPORTING DATA ──────────────────────────────────────
fund_full  = pd.read_csv(os.path.join(DATA_DIR, 'fundamental_scores_full.csv'))
prefilt_df = pd.read_csv(os.path.join(DATA_DIR, 'prefilt_passed.csv'))

fund_full = fund_full.merge(
    prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
)

# ── VOLUME PROFILE ────────────────────────────────────────────
def calculate_volume_profile(df, lookback_days=252, bins=30):
    recent    = df.tail(lookback_days).copy()
    price_min = recent['Low'].min()
    price_max = recent['High'].max()
    if price_max <= price_min:
        return None
    price_bins      = np.linspace(price_min, price_max, bins + 1)
    volume_at_price = np.zeros(bins)
    for _, row in recent.iterrows():
        candle_low   = row['Low']
        candle_high  = row['High']
        candle_vol   = row['Volume']
        candle_range = candle_high - candle_low
        if candle_range == 0:
            continue
        for i in range(bins):
            bin_low      = price_bins[i]
            bin_high     = price_bins[i + 1]
            overlap_low  = max(candle_low,  bin_low)
            overlap_high = min(candle_high, bin_high)
            if overlap_high > overlap_low:
                overlap_pct          = (overlap_high - overlap_low) / candle_range
                volume_at_price[i]  += candle_vol * overlap_pct
    poc_idx   = np.argmax(volume_at_price)
    poc_price = (price_bins[poc_idx] + price_bins[poc_idx + 1]) / 2
    total_vol  = volume_at_price.sum()
    target_vol = total_vol * 0.70
    lower_idx  = poc_idx
    upper_idx  = poc_idx
    va_vol     = volume_at_price[poc_idx]
    while va_vol < target_vol:
        can_up   = upper_idx + 1 < bins
        can_down = lower_idx - 1 >= 0
        exp_up   = volume_at_price[upper_idx + 1] if can_up   else 0
        exp_down = volume_at_price[lower_idx - 1] if can_down else 0
        if not can_up and not can_down:
            break
        if exp_up >= exp_down and can_up:
            upper_idx += 1
            va_vol    += exp_up
        elif can_down:
            lower_idx -= 1
            va_vol    += exp_down
        else:
            break
    val           = (price_bins[lower_idx] + price_bins[lower_idx + 1]) / 2
    vah           = (price_bins[upper_idx] + price_bins[upper_idx + 1]) / 2
    max_vol       = volume_at_price.max()
    edge_vol      = (volume_at_price[:3].mean() + volume_at_price[-3:].mean()) / 2
    is_bell       = max_vol > edge_vol * 2.5
    current_price = df['Close'].iloc[-1]
    if current_price > vah * 1.02:   breakout_status = 'BREAKOUT_UP'
    elif current_price < val * 0.98: breakout_status = 'BREAKOUT_DOWN'
    elif current_price > vah * 0.98: breakout_status = 'AT_RESISTANCE'
    else:                            breakout_status = 'INSIDE'
    return {
        'poc_price'      : round(poc_price, 2),
        'val'            : round(val, 2),
        'vah'            : round(vah, 2),
        'is_bell'        : is_bell,
        'breakout_status': breakout_status,
        'current_price'  : round(current_price, 2),
    }

# ── MOMENTUM SCORE (exact Day 8) ─────────────────────────────
def score_momentum(df, vp):
    scores = {}
    latest = df.iloc[-1]
    close  = latest['Close']
    ema20  = latest['EMA20']
    ema50  = latest['EMA50']
    ema200 = latest['EMA200']

    if close > ema20 > ema50 > ema200:  scores['EMA'] = 30
    elif close > ema50 > ema200:        scores['EMA'] = 20
    elif close > ema200:                scores['EMA'] = 10
    else:                               scores['EMA'] = 0

    rsi = latest['RSI']
    if 50 <= rsi <= 70:    scores['RSI'] = 20
    elif 45 <= rsi < 50:   scores['RSI'] = 12
    elif 70 < rsi <= 80:   scores['RSI'] = 8
    else:                  scores['RSI'] = 0

    # Handle both MACD Hist (space) and MACD_Hist (underscore)
    if 'MACD_Hist' in df.columns:
        macd_col = 'MACD_Hist'
        macd_line_col = 'MACD'
        signal_col    = 'Signal'
    else:
        macd_col      = 'MACD Hist'
        macd_line_col = None
        signal_col    = None

    hist = latest[macd_col]
    if macd_line_col and signal_col:
        macd   = latest[macd_line_col]
        signal = latest[signal_col]
        if macd > signal and hist > 0 and macd > 0:  scores['MACD'] = 20
        elif macd > signal and hist > 0:             scores['MACD'] = 15
        elif macd > signal:                          scores['MACD'] = 10
        else:                                        scores['MACD'] = 0
    else:
        if hist > 0:      scores['MACD'] = 20
        elif hist > -0.5: scores['MACD'] = 10
        else:             scores['MACD'] = 0

    adx      = latest['ADX']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    if adx > 25 and di_plus > di_minus:   scores['ADX'] = 15
    elif adx > 20 and di_plus > di_minus:  scores['ADX'] = 10
    elif adx > 25:                         scores['ADX'] = 5
    else:                                  scores['ADX'] = 0

    if vp:
        status = vp['breakout_status']
        if status == 'BREAKOUT_UP':     scores['VP'] = 15
        elif status == 'AT_RESISTANCE': scores['VP'] = 10
        elif status == 'INSIDE':        scores['VP'] = 5
        else:                           scores['VP'] = 0
    else:
        scores['VP'] = 5

    return sum(scores.values()), scores

# ── REVERSAL SCORE (exact Day 8) ─────────────────────────────
def score_reversal(df, vp, lookback=20):
    scores = {}
    if len(df) < lookback + 10:
        return 0, {}

    # Handle column name variants
    macd_col  = 'MACD_Hist' if 'MACD_Hist' in df.columns else 'MACD Hist'
    vol_col   = 'Vol_Ratio'  if 'Vol_Ratio'  in df.columns else 'Vol Ratio'

    latest_rsi = df['RSI'].iloc[-1]
    if latest_rsi < 35:    scores['RSI_Level'] = 20
    elif latest_rsi < 45:  scores['RSI_Level'] = 10
    else:                  scores['RSI_Level'] = 0

    rsi_5d    = df['RSI'].tail(5).min()
    rsi_20d   = df['RSI'].tail(20).min()
    price_5d  = df['Close'].tail(5).min()
    price_20d = df['Close'].tail(20).min()
    scores['RSI_Divergence'] = 15 if (price_5d <= price_20d and rsi_5d > rsi_20d) else 0

    latest_hist = df[macd_col].iloc[-1]
    prev_hist   = df[macd_col].iloc[-5]
    if latest_hist > 0 and prev_hist < 0:
        scores['MACD_Cross']  = 20
        scores['MACD_Rising'] = 0
    elif latest_hist > prev_hist and latest_hist < 0:
        scores['MACD_Cross']  = 0
        scores['MACD_Rising'] = 15
    else:
        scores['MACD_Cross']  = 0
        scores['MACD_Rising'] = 0

    macd_5d  = df[macd_col].tail(5).min()
    macd_20d = df[macd_col].tail(20).min()
    scores['MACD_Divergence'] = 15 if (price_5d <= price_20d and macd_5d > macd_20d) else 0

    adx_latest = df['ADX'].iloc[-1]
    adx_10d    = df['ADX'].iloc[-10]
    if adx_latest < adx_10d and adx_latest > 20:  scores['ADX'] = 15
    elif adx_latest < adx_10d:                    scores['ADX'] = 8
    else:                                          scores['ADX'] = 0

    vol_ratio = df[vol_col].iloc[-1]
    if vol_ratio < 0.6:    scores['Volume'] = 15
    elif vol_ratio < 0.8:  scores['Volume'] = 8
    else:                  scores['Volume'] = 0

    return min(sum(scores.values()), 100), scores

# ── DYNAMIC CONSOLIDATION DETECTION (exact Day 8) ────────────
def detect_consolidation_dynamic(df, min_days=300, max_range_pct=50,
                                  max_total_drift=10, inside_pct_required=0.82):
    close = df['Close']
    n     = len(df)
    if n < min_days + 5:
        return {'found': False, 'is_valid': False, 'consol_days': 0}
    best_result = None
    for lookback in range(min_days, min(800, n - 5)):
        start_idx  = n - lookback
        segment    = df.iloc[start_idx:n]
        range_high = segment['High'].quantile(0.95)
        range_low  = segment['Low'].quantile(0.05)
        range_pct  = (range_high - range_low) / range_low * 100
        rng        = range_high - range_low
        if range_pct > max_range_pct or rng == 0:
            continue
        buffer      = 0.03
        bars_inside = (
            (segment['High'] <= range_high * (1 + buffer)) &
            (segment['Low']  >= range_low  * (1 - buffer))
        ).sum()
        pct_inside = bars_inside / len(segment)
        if pct_inside < inside_pct_required:
            continue
        x        = np.arange(len(segment))
        hi_drift = abs(np.polyfit(x, segment['High'].values,  1)[0] / segment['High'].mean()  * 100) * lookback
        lo_drift = abs(np.polyfit(x, segment['Low'].values,   1)[0] / segment['Low'].mean()   * 100) * lookback
        cl_drift = abs(np.polyfit(x, segment['Close'].values, 1)[0] / segment['Close'].mean() * 100) * lookback
        if hi_drift > max_total_drift or lo_drift > max_total_drift or cl_drift > max_total_drift:
            continue
        price_start  = segment['Close'].iloc[0]
        price_end    = segment['Close'].iloc[-1]
        price_change = (price_end - price_start) / price_start * 100
        if abs(price_change) > 25:
            continue
        end_vs_range = (price_end - range_low) / rng * 100
        if end_vs_range < -10:
            continue
        close_arr    = segment['Close'].values
        peak_idx     = np.argmax(close_arr)
        peak_pos     = peak_idx / lookback * 100
        peak_price   = close_arr.max()
        peak_vs_rng  = (peak_price  - range_low) / rng * 100
        start_vs_rng = (price_start - range_low) / rng * 100
        if peak_pos > 20 and peak_pos < 80 and start_vs_rng < 35 and end_vs_range < 35 and peak_vs_rng > 90:
            continue
        pre_start   = max(0, start_idx - 120)
        pre_segment = df.iloc[pre_start:start_idx]
        pre_close   = pre_segment['Close'].values
        if len(pre_close) > 5:
            pre_x    = np.arange(len(pre_close))
            pre_norm = np.polyfit(pre_x, pre_close, 1)[0] / pre_close.mean() * 100
            if pre_norm < (-0.15 if lookback < 365 else -0.60):
                continue
            pre_high = pre_segment['High'].max()
            drop_pct = (range_high - pre_high) / pre_high * 100
            if drop_pct < (-20 if lookback < 365 else -60):
                continue
        if lookback > (best_result['consol_days'] if best_result else 0):
            best_result = {
                'consol_days': lookback,
                'range_high' : round(range_high, 2),
                'range_low'  : round(range_low,  2),
                'range_pct'  : round(range_pct,  2),
                'pct_inside' : round(pct_inside,  3),
                'start_idx'  : start_idx,
            }
    if best_result is None:
        return {'found': False, 'is_valid': False, 'consol_days': 0}

    consol_days   = best_result['consol_days']
    range_high    = best_result['range_high']
    range_low     = best_result['range_low']
    current_price = close.iloc[-1]
    consol_df     = df.iloc[best_result['start_idx']:n]
    res_touches   = int((consol_df['High'] >= range_high * 0.98).sum())
    sup_touches   = int((consol_df['Low']  <= range_low  * 1.02).sum())
    pct_above     = (current_price - range_high) / range_high * 100

    vol_col = 'Vol_Ratio' if 'Vol_Ratio' in df.columns else 'Vol Ratio'
    vol_ratio = round(df[vol_col].tail(5).max(), 2)

    vp       = calculate_volume_profile(consol_df, lookback_days=consol_days, bins=20)
    has_bell = vp['is_bell'] if vp else False

    if consol_days < 365:   duration_label = f'{consol_days}d (under 1 year)'
    elif consol_days < 500: duration_label = f'{consol_days}d (1-1.5 years)'
    elif consol_days < 730: duration_label = f'{consol_days}d (1.5-2 years)'
    else:                   duration_label = f'{consol_days}d (2+ years)'

    is_valid = res_touches >= 2 and sup_touches >= 2
    return {
        'found'             : True,
        'is_valid'          : is_valid,
        'consol_days'       : consol_days,
        'duration_label'    : duration_label,
        'range_high'        : range_high,
        'range_low'         : range_low,
        'range_pct'         : best_result['range_pct'],
        'pct_above'         : round(pct_above, 2),
        'is_breaking_out'   : current_price >= range_high * 0.98,
        'breakout_volume'   : vol_ratio,
        'has_bell'          : has_bell,
        'current_price'     : round(current_price, 2),
        'resistance_touches': res_touches,
        'support_touches'   : sup_touches,
    }

def get_consolidation_info(df):
    result = detect_consolidation_dynamic(df)
    if not result.get('found') or not result.get('is_valid'):
        return {
            'consolidating'     : False,
            'consol_days'       : 0,
            'duration_label'    : 'No consolidation',
            'range_high'        : None,
            'range_low'         : None,
            'range_pct'         : None,
            'pct_to_breakout'   : None,
            'is_breaking_out'   : False,
            'breakout_volume'   : None,
            'has_bell'          : False,
            'resistance_touches': 0,
            'support_touches'   : 0,
            'current_price'     : df['Close'].iloc[-1],
        }
    return {
        'consolidating'     : True,
        'consol_days'       : result['consol_days'],
        'duration_label'    : result['duration_label'],
        'range_high'        : result['range_high'],
        'range_low'         : result['range_low'],
        'range_pct'         : result['range_pct'],
        'pct_to_breakout'   : result['pct_above'],
        'is_breaking_out'   : result['is_breaking_out'],
        'breakout_volume'   : result['breakout_volume'],
        'has_bell'          : result['has_bell'],
        'resistance_touches': result['resistance_touches'],
        'support_touches'   : result['support_touches'],
        'current_price'     : result['current_price'],
    }

def classify_mcap(mcap_cr):
    if mcap_cr >= 20000:  return 'Large Cap'
    elif mcap_cr >= 5000: return 'Mini Large Cap'
    elif mcap_cr >= 1000: return 'Mid Cap'
    else:                 return 'Small Cap'

# ── LOAD CHECKPOINT IF EXISTS ─────────────────────────────────
if os.path.exists(TECH_CHECKPOINT):
    with open(TECH_CHECKPOINT, 'rb') as f:
        ckpt = pickle.load(f)
    tech_reports = ckpt['reports']
    done_syms    = ckpt['done']
    print(f"Resuming from checkpoint — {len(done_syms)} already done")
else:
    tech_reports = []
    done_syms    = set()

remaining = [s for s in symbols
             if s in indicator_data and s not in done_syms]
print(f"Stocks to process : {len(remaining)}")
print(f"(Note: VP + consolidation detection makes this ~30-60 mins)")
print("-" * 60)

# ── MAIN LOOP ─────────────────────────────────────────────────
for i, symbol in enumerate(remaining):
    try:
        df       = indicator_data[symbol]
        latest   = df.iloc[-1]
        fund_row = fund_full[fund_full['Symbol'] == symbol]

        fund_score = float(fund_row['Final Score'].values[0]) \
                     if len(fund_row) > 0 and 'Final Score' in fund_row.columns \
                     else 50.0
        sector     = fund_row['Sector'].values[0] \
                     if len(fund_row) > 0 else 'Unknown'
        mcap_cr    = float(fund_row['Market_Cap_Cr'].values[0]) \
                     if len(fund_row) > 0 and 'Market_Cap_Cr' in fund_row.columns \
                     else 0

        close    = latest['Close']
        rsi      = round(latest['RSI'],    2)
        adx      = round(latest['ADX'],    2)
        ema20    = round(latest['EMA20'],  2)
        ema50    = round(latest['EMA50'],  2)
        ema200   = round(latest['EMA200'], 2)
        di_plus  = round(latest['DI_Plus'],  2)
        di_minus = round(latest['DI_Minus'], 2)

        # Handle column name variants
        macd_col = 'MACD_Hist' if 'MACD_Hist' in df.columns else 'MACD Hist'
        vol_col  = 'Vol_Ratio'  if 'Vol_Ratio'  in df.columns else 'Vol Ratio'
        hist     = round(latest[macd_col], 4)
        vol_r    = round(latest[vol_col],  2)

        vp            = calculate_volume_profile(df)
        mom_score, _  = score_momentum(df, vp)
        rev_score, _  = score_reversal(df, vp)
        consol_info   = get_consolidation_info(df)

        # ── BEST SETUP (exact Day 8 logic) ────────────────────
        if mom_score >= rev_score and mom_score >= 50:
            best_setup = 'Momentum'
            tech_score = mom_score
        elif rev_score >= mom_score and rev_score >= 50:
            best_setup = 'Reversal'
            tech_score = rev_score
        else:
            best_setup = 'Watching'
            tech_score = max(mom_score, rev_score)

        in_consol       = consol_info['consolidating']
        consol_days     = consol_info['consol_days']
        consol_label    = consol_info['duration_label']
        pct_to_breakout = consol_info['pct_to_breakout'] or 0
        consol_volume   = consol_info['breakout_volume']  or 0
        cap_category    = classify_mcap(mcap_cr)

        # ── TIER (exact Day 8 logic) ──────────────────────────
        if fund_score >= 60 and tech_score >= 65 and best_setup != 'Watching':
            tier = f'TIER 1 — BUY NOW ({best_setup})'
        elif fund_score >= 60 and in_consol and pct_to_breakout > -5:
            tier = 'TIER 1 — BREAKOUT IMMINENT'
        elif fund_score >= 60 and tech_score >= 40 and best_setup != 'Watching':
            tier = 'TIER 2 — WATCHLIST'
        elif fund_score >= 60 and in_consol:
            tier = 'TIER 2 — BASE BUILDING'
        else:
            tier = 'TIER 3 — WAITING'

        tech_reports.append({
            'Symbol'          : symbol,
            'Sector'          : sector,
            'Fund Score'      : fund_score,
            'Market Cap Cr'   : round(mcap_cr, 2),
            'Cap Category'    : cap_category,
            'Current Price'   : round(close,   2),
            'RSI'             : rsi,
            'ADX'             : adx,
            'MACD Hist'       : hist,
            'Vol Ratio'       : vol_r,
            'EMA20'           : ema20,
            'EMA50'           : ema50,
            'EMA200'          : ema200,
            'DI_Plus'         : di_plus,
            'DI_Minus'        : di_minus,
            'Momentum Score'  : mom_score,
            'Reversal Score'  : rev_score,
            'Best Setup'      : best_setup,
            'Tech Score'      : tech_score,
            'Tier'            : tier,
            'In Consolidation': in_consol,
            'Consol Days'     : consol_days,
            'Consol Label'    : consol_label,
            'Pct to Breakout' : round(pct_to_breakout, 2),
            'Consol Volume'   : consol_volume,
        })
        done_syms.add(symbol)

    except Exception as e:
        pass

    if (i + 1) % 50 == 0 or (i + 1) == len(remaining):
        pct = (i + 1) / len(remaining) * 100
        print(f"  [{i+1:4d}/{len(remaining)}] {pct:5.1f}% | "
              f"Done: {len(tech_reports)} | "
              f"Time: {datetime.now().strftime('%H:%M:%S')}")
        with open(TECH_CHECKPOINT, 'wb') as f:
            pickle.dump({'reports': tech_reports, 'done': done_syms}, f)

# ── MERGE ML SCORES ───────────────────────────────────────────
tech_df       = pd.DataFrame(tech_reports)
ml_scores_df  = pd.DataFrame(all_ml_scores)

ml_merge_cols = [
    'Symbol', 'ML_Prediction', 'ML_Confidence',
    'Forecast_25d_Pct', 'Forecast_45d_Pct', 'Forecast_180d_Pct',
    'Forecast_25d_Price', 'Forecast_45d_Price', 'Forecast_180d_Price',
    'Bottom_Rev_Prob', 'Top_Rev_Prob', 'Bottom_Rev_Flag', 'Top_Rev_Flag',
    'Bullish_Cont_Prob', 'Bearish_Cont_Prob',
]
tech_final = tech_df.merge(ml_scores_df[ml_merge_cols], on='Symbol', how='left')

# ── SAVE ──────────────────────────────────────────────────────
tech_final.to_csv(os.path.join(DATA_DIR, 'technical_report_full.csv'), index=False)
ml_scores_df.to_csv(os.path.join(DATA_DIR, 'ml_scores_full.csv'), index=False)

if os.path.exists(TECH_CHECKPOINT):
    os.remove(TECH_CHECKPOINT)

print(f"\n✅ Cell 9 — Technical analysis complete")
print(f"   Stocks analysed  : {len(tech_final)}")
print(f"   Columns          : {len(tech_final.columns)}")
print(f"\n   Tier distribution:")
print(tech_final['Tier'].value_counts().to_string())
print(f"\n   Cap distribution:")
print(tech_final['Cap Category'].value_counts().to_string())
print(f"\n   Tech Score sample (first 5):")
print(tech_final[['Symbol','Best Setup','Tech Score','Tier']].head(10).to_string(index=False))

Running full technical analysis...
Started : 23:45:24
------------------------------------------------------------
Stocks to process : 752
(Note: VP + consolidation detection makes this ~30-60 mins)
------------------------------------------------------------
  [  50/752]   6.6% | Done: 50 | Time: 23:45:48
  [ 100/752]  13.3% | Done: 100 | Time: 23:46:16
  [ 150/752]  19.9% | Done: 150 | Time: 23:46:43
  [ 200/752]  26.6% | Done: 200 | Time: 23:47:06
  [ 250/752]  33.2% | Done: 250 | Time: 23:47:31
  [ 300/752]  39.9% | Done: 300 | Time: 23:47:55
  [ 350/752]  46.5% | Done: 350 | Time: 23:48:18
  [ 400/752]  53.2% | Done: 400 | Time: 23:48:41
  [ 450/752]  59.8% | Done: 450 | Time: 23:49:07
  [ 500/752]  66.5% | Done: 500 | Time: 23:49:35
  [ 550/752]  73.1% | Done: 550 | Time: 23:49:56
  [ 600/752]  79.8% | Done: 600 | Time: 23:50:21
  [ 650/752]  86.4% | Done: 650 | Time: 23:50:47
  [ 700/752]  93.1% | Done: 700 | Time: 23:51:10
  [ 750/752]  99.7% | Done: 750 | Time: 23:51:17
  [ 75

In [20]:
# ── CELL 10A: WEEKLY REPORT DATA PREP ────────────────────────
# Direct automation of Day 8 Cell 13
# Computes: SectorScore, CapScore, RankJumpers,
#           Vol5D, BreakoutVol, VolumeInference, SectorTrend
#           BreakoutTracker

print("Preparing weekly report data...")
print("-" * 60)

# ── LOAD TECH REPORT ──────────────────────────────────────────
tech_df    = pd.read_csv(os.path.join(DATA_DIR, 'technical_report_full.csv'))
fund_full  = pd.read_csv(os.path.join(DATA_DIR, 'fundamental_scores_full.csv'))
prefilt_df = pd.read_csv(os.path.join(DATA_DIR, 'prefilt_passed.csv'))

print(f"  Loaded tech_df: {len(tech_df)} rows, {len(tech_df.columns)} cols")

# ── STEP 1: SECTOR SCORE + CAP SCORE (exact Day 8 method) ────
# Score = fund score relative to best in same sector/cap (not percentile)
fund_full = fund_full.merge(
    prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
)

def classify_mcap_score(mcap_cr):
    if mcap_cr >= 20000:  return 'Large Cap'
    elif mcap_cr >= 5000: return 'Mini Large Cap'
    elif mcap_cr >= 1000: return 'Mid Cap'
    else:                 return 'Small Cap'

fund_full['Cap Category'] = fund_full['Market_Cap_Cr'].apply(
    lambda x: classify_mcap_score(x or 0)
)

# Sector Score: fund score / max fund score in same sector * 10
fund_full['Sector Score'] = 0.0
for sector in fund_full['Sector'].unique():
    mask      = fund_full['Sector'] == sector
    max_score = fund_full[mask]['Final Score'].max()
    if max_score > 0:
        fund_full.loc[mask, 'Sector Score'] = (
            fund_full[mask]['Final Score'] / max_score * 10
        ).round(1)

# Cap Score: fund score / max fund score in same sector + cap bucket
fund_full['Cap Score'] = 0.0
for sector in fund_full['Sector'].unique():
    for cap in ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']:
        mask   = (fund_full['Sector'] == sector) & (fund_full['Cap Category'] == cap)
        subset = fund_full[mask]
        if len(subset) == 0:
            continue
        max_score = subset['Final Score'].max()
        if max_score > 0:
            fund_full.loc[mask, 'Cap Score'] = (
                subset['Final Score'] / max_score * 10
            ).round(1)

# Drop old score columns and merge fresh
for col in ['Sector Score', 'Cap Score', 'Sector Score_x', 'Cap Score_x',
            'Sector Score_y', 'Cap Score_y']:
    if col in tech_df.columns:
        tech_df = tech_df.drop(columns=[col])

tech_df = tech_df.merge(
    fund_full[['Symbol', 'Sector Score', 'Cap Score']],
    on='Symbol', how='left'
)
print(f"  Sector + Cap scores merged: {tech_df['Sector Score'].notna().sum()} stocks")

# ── STEP 2: RANK JUMPERS ──────────────────────────────────────
SCORES_FILE  = os.path.join(DATA_DIR, 'last_week_scores.csv')
rank_jumpers = set()

current_scores = tech_df[['Symbol', 'Sector Score', 'Cap Score']].copy()

if os.path.exists(SCORES_FILE):
    last_scores   = pd.read_csv(SCORES_FILE)
    merged_scores = current_scores.merge(
        last_scores[['Symbol', 'Sector Score', 'Cap Score']],
        on='Symbol', how='left', suffixes=('_now', '_prev')
    )
    for _, row in merged_scores.iterrows():
        prev_sec = row.get('Sector Score_prev', 0) or 0
        prev_cap = row.get('Cap Score_prev',    0) or 0
        curr_sec = row.get('Sector Score_now',  0) or 0
        curr_cap = row.get('Cap Score_now',     0) or 0
        sec_jump = (curr_sec - prev_sec) / prev_sec * 100 if prev_sec > 0 else 0
        cap_jump = (curr_cap - prev_cap) / prev_cap * 100 if prev_cap > 0 else 0
        if sec_jump >= 10 or cap_jump >= 10:
            rank_jumpers.add(row['Symbol'])
    print(f"  Rank jumpers: {len(rank_jumpers)}")
else:
    print(f"  No previous scores — jumpers tracked from next week")

current_scores.to_csv(SCORES_FILE, index=False)
print(f"  Scores saved for next week")

# ── STEP 3: VOL 5D RATIO ─────────────────────────────────────
vol_5d_data = {}
for symbol, df in price_data.items():
    try:
        if len(df) < 20:
            vol_5d_data[symbol] = 1.0
            continue
        vol_5d_avg      = df['Volume'].iloc[-5:].mean()
        vol_prior15_avg = df['Volume'].iloc[-20:-5].mean()
        vol_5d_data[symbol] = round(
            vol_5d_avg / vol_prior15_avg if vol_prior15_avg > 0 else 1.0, 2
        )
    except:
        vol_5d_data[symbol] = 1.0

tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data).fillna(1.0)
print(f"  Vol 5D Ratio computed")

# ── STEP 4: BREAKOUT VOL ─────────────────────────────────────
breakout_vol_data = {}
for _, row in tech_df[tech_df['In Consolidation'] == True].iterrows():
    symbol      = row['Symbol']
    consol_days = int(row['Consol Days'])
    if symbol not in price_data or consol_days < 5:
        continue
    df = price_data[symbol]
    if len(df) < consol_days + 5:
        breakout_vol_data[symbol] = 1.0
        continue
    week_vol   = df['Volume'].iloc[-5:].mean()
    consol_avg = df['Volume'].iloc[-consol_days:-5].mean()
    breakout_vol_data[symbol] = round(
        week_vol / consol_avg if consol_avg > 0 else 1.0, 2
    )

tech_df['Breakout Vol'] = tech_df['Symbol'].map(breakout_vol_data).fillna(0.0)
print(f"  Breakout Vol computed")

# ── STEP 5: VOLUME INFERENCE ──────────────────────────────────
def get_volume_inference(row):
    vol   = float(row.get('Vol 5D Ratio', 1.0))
    setup = str(row['Best Setup'])
    adx   = float(row['ADX'])
    if   vol >= 3.0: vol_label = 'Extremely High'
    elif vol >= 2.0: vol_label = 'Very High'
    elif vol >= 1.5: vol_label = 'High'
    elif vol >= 1.0: vol_label = 'Normal'
    elif vol >= 0.7: vol_label = 'Low'
    else:            vol_label = 'Very Low'

    if setup == 'Momentum':
        if   vol >= 2.0: inf = 'Strong participation → trend likely to continue'
        elif vol >= 1.5: inf = 'Good volume support → momentum confirmed'
        elif vol >= 1.0: inf = 'Average volume → monitor for increase'
        elif vol >= 0.7: inf = 'Weak volume → momentum may fade, wait'
        else:            inf = 'Very low volume → no conviction, caution'
    elif setup == 'Reversal':
        if   vol <= 0.5: inf = 'Sellers exhausted → reversal likely'
        elif vol <= 0.7: inf = 'Low volume on down move → selling pressure easing'
        elif vol <= 1.0: inf = 'Neutral volume → wait for low-volume base to form'
        elif vol <= 1.5: inf = 'Elevated volume on reversal → watch direction'
        else:            inf = 'High volume → could be capitulation or distribution'
    else:
        if   adx > 40 and vol >= 2.0: inf = 'Above avg volume but no setup → watch closely'
        elif adx > 40:                inf = 'Strong downtrend → no entry signal'
        elif vol <= 0.5:              inf = 'Very low volume → no interest, wait'
        elif vol >= 1.5 and adx < 35: inf = 'Above avg volume but no setup → watch'
        elif vol >= 1.0:              inf = 'Normal activity → no setup forming yet'
        else:                         inf = 'Low activity → no setup forming yet'
    return vol_label, inf

tech_df['Vol Label']     = ''
tech_df['Vol Inference'] = ''
for idx, row in tech_df.iterrows():
    label, inf = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inf
print(f"  Volume inference computed")

# ── STEP 6: SECTOR TREND (exact Day 8 method) ─────────────────
SECTOR_INDEX_MAP = {
    'Information Technology'            : '^CNXIT',
    'Healthcare'                        : '^CNXPHARMA',
    'Financial Services'                : 'FINIETF.NS',
    'Capital Goods'                     : '^CNXINFRA',
    'Chemicals'                         : 'MID150BEES.NS',
    'Consumer Durables'                 : '^CNXCONSUM',
    'Oil, Gas & Consumable Fuels'       : '^CNXENERGY',
    'Automobile and Auto Components'    : '^CNXAUTO',
    'Textiles'                          : 'MID150BEES.NS',
    'Consumer Services'                 : 'MID150BEES.NS',
    'Banking'                           : '^NSEBANK',
    'Fast Moving Consumer Goods'        : '^CNXFMCG',
    'Metals & Mining'                   : '^CNXMETAL',
    'Realty'                            : '^CNXREALTY',
    'Services'                          : '^CNXSERVICE',
    'Construction'                      : '^CNXINFRA',
    'Power'                             : '^CNXENERGY',
    'Pharmaceuticals'                   : '^CNXPHARMA',
    'Cement'                            : '^CNXCMDT',
    'Defence'                           : 'MID150BEES.NS',
    'Telecommunication'                 : 'MID150BEES.NS',
    'Diversified'                       : 'MID150BEES.NS',
    'Utilities'                         : '^CNXENERGY',
    'Media Entertainment & Publication' : 'MID150BEES.NS',
    'Agriculture'                       : 'MID150BEES.NS',
    'Forest Materials'                  : 'MID150BEES.NS',
}

print(f"  Fetching sector index data...")
ticker_data = {}
for ticker in set(SECTOR_INDEX_MAP.values()):
    try:
        df = yf.download(ticker, period='1y', interval='1d',
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]
        if len(df) > 50:
            ticker_data[ticker] = df
    except:
        pass

fallback     = 'MID150BEES.NS'
sector_price = {}
for sector, ticker in SECTOR_INDEX_MAP.items():
    if ticker in ticker_data:
        sector_price[sector] = ticker_data[ticker]
    elif fallback in ticker_data:
        sector_price[sector] = ticker_data[fallback]

def calculate_sector_indicators(df):
    data = df.copy()
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()
    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    rs       = gain.ewm(span=14, adjust=False).mean() / (
                loss.ewm(span=14, adjust=False).mean() + 1e-9)
    data['RSI']       = 100 - (100 / (1 + rs))
    ema12             = data['Close'].ewm(span=12, adjust=False).mean()
    ema26             = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']      = ema12 - ema26
    data['Signal']    = data['MACD'].ewm(span=9,  adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']
    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr       = pd.concat([high - low, (high - close.shift()).abs(),
                          (low - close.shift()).abs()], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus + 1e-9)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus
    return data

def get_sector_trend(sector):
    if sector not in sector_price:
        return 'No data', '—'
    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]
    close    = latest['Close']
    rsi      = round(latest['RSI'],   1)
    adx      = round(latest['ADX'],   1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']
    above_ema200 = close > ema200
    above_ema50  = close > ema50
    ema_aligned  = ema20 > ema50 > ema200
    ema_bearish  = ema20 < ema50 < ema200
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    bull_score   = sum([above_ema200, above_ema50, ema_aligned,
                        uptrend, macd_bull, rsi > 55])
    bear_score   = sum([not above_ema200, not above_ema50, ema_bearish,
                        not uptrend, macd_bear, rsi < 45])
    if   bull_score >= 5 and adx > 25:  label = 'Strong Uptrend ↑↑'
    elif bull_score >= 4 and adx > 20:  label = 'Uptrend ↑'
    elif bull_score >= 3 and adx <= 20: label = 'Weak Uptrend →↑'
    elif bull_score == bear_score:      label = 'Sideways →'
    elif bear_score >= 5 and adx > 25:  label = 'Strong Downtrend ↓↓'
    elif bear_score >= 4 and adx > 20:  label = 'Downtrend ↓'
    elif bear_score >= 3 and adx <= 20: label = 'Weak Downtrend →↓'
    else:                               label = 'Sideways →'
    ema_str  = '↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'
    macd_str = '▲' if macd_bull else '▼' if macd_bear else '~'
    detail   = (f"RSI {rsi} | ADX {adx} | MACD {macd_str} | "
                f"EMA {ema_str} | {'Above' if above_ema200 else 'Below'} EMA200")
    return label, detail

tech_df['Sector Trend']  = ''
tech_df['Sector Detail'] = ''
for idx, row in tech_df.iterrows():
    s_label, s_detail = get_sector_trend(str(row['Sector']))
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail
print(f"  Sector trends computed")

# ── STEP 7: BREAKOUT TRACKER (exact Day 8 method) ────────────
BREAKOUT_FILE = os.path.join(DATA_DIR, 'breakout_tracker.csv')
today_str     = datetime.now().strftime('%Y-%m-%d')

if os.path.exists(BREAKOUT_FILE):
    bk_df = pd.read_csv(BREAKOUT_FILE)
else:
    bk_df = pd.DataFrame(columns=[
        'Symbol', 'First_Breakout_Date', 'Weeks_Count',
        'Last_Pct', 'Cap_Category', 'Sector', 'Fund_Score'
    ])

# Current breakouts: stocks above range high (Pct to Breakout > 0)
current_breakouts = tech_df[
    (tech_df['In Consolidation'] == True) &
    (tech_df['Pct to Breakout'] > 0)
][['Symbol', 'Pct to Breakout', 'Cap Category',
   'Sector', 'Fund Score', 'Current Price',
   'Vol 5D Ratio', 'RSI', 'Tech Score']].copy()

updated_rows = []
for _, row in current_breakouts.iterrows():
    sym      = row['Symbol']
    existing = bk_df[bk_df['Symbol'] == sym]
    if len(existing) > 0:
        weeks = int(existing.iloc[0]['Weeks_Count']) + 1
        updated_rows.append({
            'Symbol'             : sym,
            'First_Breakout_Date': existing.iloc[0]['First_Breakout_Date'],
            'Weeks_Count'        : weeks,
            'Last_Pct'           : round(row['Pct to Breakout'], 2),
            'Cap_Category'       : row['Cap Category'],
            'Sector'             : row['Sector'],
            'Fund_Score'         : row['Fund Score'],
        })
    else:
        updated_rows.append({
            'Symbol'             : sym,
            'First_Breakout_Date': today_str,
            'Weeks_Count'        : 1,
            'Last_Pct'           : round(row['Pct to Breakout'], 2),
            'Cap_Category'       : row['Cap Category'],
            'Sector'             : row['Sector'],
            'Fund_Score'         : row['Fund Score'],
        })

# Keep stocks tracked < 3 weeks that are still near breakout
for _, row in bk_df.iterrows():
    sym           = row['Symbol']
    already_added = any(r['Symbol'] == sym for r in updated_rows)
    if not already_added and int(row['Weeks_Count']) < 3:
        tech_row = tech_df[tech_df['Symbol'] == sym]
        if len(tech_row) > 0:
            pct = float(tech_row.iloc[0]['Pct to Breakout'])
            if pct > -5:
                updated_rows.append({
                    'Symbol'             : sym,
                    'First_Breakout_Date': row['First_Breakout_Date'],
                    'Weeks_Count'        : int(row['Weeks_Count']),
                    'Last_Pct'           : round(pct, 2),
                    'Cap_Category'       : row['Cap_Category'],
                    'Sector'             : row['Sector'],
                    'Fund_Score'         : row['Fund_Score'],
                })

new_bk_df = pd.DataFrame(updated_rows)
new_bk_df.to_csv(BREAKOUT_FILE, index=False)
print(f"  Breakout tracker: {len(new_bk_df)} stocks")

# Merge breakout tracker into tech_df
if len(new_bk_df) > 0:
    tech_df = tech_df.merge(
        new_bk_df[['Symbol', 'Weeks_Count', 'First_Breakout_Date']],
        on='Symbol', how='left'
    )
else:
    tech_df['Weeks_Count']         = None
    tech_df['First_Breakout_Date'] = None

# ── STEP 8: SAVE ENRICHED TECH REPORT ────────────────────────
tech_df.to_csv(os.path.join(DATA_DIR, 'technical_report_full.csv'), index=False)

print(f"\n✅ Cell 10A — Weekly report data prep complete")
print(f"   Rows    : {len(tech_df)}")
print(f"   Columns : {len(tech_df.columns)}")
print(f"\n   Sample Tier 1 Momentum stocks:")
t1 = tech_df[tech_df['Tier'].str.contains('TIER 1', na=False)].sort_values(
    'Fund Score', ascending=False)
print(t1[['Symbol','Fund Score','Tech Score','Sector Score',
          'Cap Score','Tier']].head(10).to_string(index=False))

Preparing weekly report data...
------------------------------------------------------------
  Loaded tech_df: 752 rows, 39 cols
  Sector + Cap scores merged: 752 stocks
  Rank jumpers: 0
  Scores saved for next week
  Vol 5D Ratio computed
  Breakout Vol computed
  Volume inference computed
  Fetching sector index data...
  Sector trends computed
  Breakout tracker: 23 stocks

✅ Cell 10A — Weekly report data prep complete
   Rows    : 752
   Columns : 49

   Sample Tier 1 Momentum stocks:
    Symbol  Fund Score  Tech Score  Sector Score  Cap Score                        Tier
 SOLARINDS        89.0          75          10.0       10.0 TIER 1 — BUY NOW (Momentum)
 NAM-INDIA        86.0          75          10.0       10.0 TIER 1 — BUY NOW (Momentum)
  LLOYDSME        84.0          88          10.0       10.0 TIER 1 — BUY NOW (Momentum)
NATCOPHARM        84.0         100           9.7        9.9 TIER 1 — BUY NOW (Momentum)
   SANDUMA        83.7          65          10.0       10.0 TIER 

In [22]:
# ── CELL 10B: GENERATE WEEKLY TECHNICAL REPORT ───────────────
# Direct automation of Day 8 Cell 15

CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']
today_str = datetime.now().strftime('%Y-%m-%d')

# ── HELPER FUNCTIONS (exact Day 8) ───────────────────────────
def get_reason(row):
    setup  = str(row['Best Setup'])
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    macd   = float(row['MACD Hist'])
    price  = float(row['Current Price'])
    ema20  = float(row['EMA20'])
    ema50  = float(row['EMA50'])
    ema200 = float(row['EMA200'])
    above_ema200 = price > ema200
    ema_aligned  = ema20 > ema50 > ema200
    if setup == 'Momentum':
        if ema_aligned and rsi > 55:      return f'Strong uptrend, EMA aligned, RSI {rsi:.0f}'
        elif above_ema200 and rsi > 50:   return f'Momentum building, above EMA200, RSI {rsi:.0f}'
        elif above_ema200 and rsi <= 50:  return f'Consolidating above EMA200, RSI {rsi:.0f}'
        elif not above_ema200 and rsi>60: return f'Short bounce, below EMA200, RSI {rsi:.0f}'
        else:                             return f'Weak momentum, below EMA200, RSI {rsi:.0f}'
    elif setup == 'Reversal':
        if rsi < 30:   return f'Deeply oversold RSI {rsi:.0f}, reversal imminent'
        elif rsi < 35: return f'Oversold RSI {rsi:.0f}, watch for turn'
        elif macd > 0: return f'RSI {rsi:.0f}, MACD turning up'
        else:          return f'Divergence forming, RSI {rsi:.0f}'
    else:
        if adx > 40:   return f'Strong downtrend ADX {adx:.0f}, wait'
        elif adx > 25: return f'Downtrend ADX {adx:.0f}, no setup yet'
        elif rsi < 30: return f'Deeply oversold RSI {rsi:.0f}, watch for turn'
        elif rsi < 40: return f'Oversold RSI {rsi:.0f}, watch for turn'
        else:          return f'RSI {rsi:.0f}, setup not confirmed'

def get_breakout_vol_inference(week_vol, break_vol):
    if   break_vol >= 2.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡  → Volume building toward breakout"
    elif break_vol >= 1.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~  → Normal volume, no breakout pressure"
    elif break_vol >= 0.7: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Quiet, below consolidation avg"
    else:                  return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Very quiet, no breakout interest"

def passes_rank_filter(row):
    return (float(row.get('Cap Score',    0) or 0) >= 7 or
            float(row.get('Sector Score', 0) or 0) >= 7)

def is_rank_jumper(row):
    return row['Symbol'] in rank_jumpers

def cap_order(x):
    return CAP_ORDER.index(x) if x in CAP_ORDER else 99

def mcap_str(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 100000: return f"Rs{v/100000:.1f}L Cr"
        return f"Rs{v:,.0f}Cr"
    except:
        return "Rs—"

# ── PREPARE DATA SLICES (exact Day 8) ────────────────────────
# Tier 1 Momentum
tier1_mom = tech_df[
    tech_df['Tier'].str.startswith('TIER 1') &
    (tech_df['Best Setup'] == 'Momentum')
].copy()
tier1_mom['_cap_order'] = tier1_mom['Cap Category'].apply(cap_order)
tier1_mom = tier1_mom.sort_values(
    ['_cap_order', 'Fund Score'], ascending=[True, False]
).reset_index(drop=True)

# Tier 2A Watchlist — rank filtered, top 25
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a[
    tier2a.apply(passes_rank_filter, axis=1) |
    tier2a['Symbol'].isin(rank_jumpers)
].sort_values('Fund Score', ascending=False).head(25).reset_index(drop=True)

# Tier 2B Base Building — rank filtered, top 25
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b[
    tier2b.apply(passes_rank_filter, axis=1) |
    tier2b['Symbol'].isin(rank_jumpers)
].sort_values('Fund Score', ascending=False).head(25).reset_index(drop=True)

# Tier 3 Momentum — rank filtered (long only)
tier3_mom = tech_df[
    (tech_df['Tier'] == 'TIER 3 — WAITING') &
    (tech_df['Best Setup'] == 'Momentum')
].copy()
tier3_mom['_cap_order'] = tier3_mom['Cap Category'].apply(cap_order)
tier3_mom = tier3_mom[
    tier3_mom.apply(passes_rank_filter, axis=1) |
    tier3_mom['Symbol'].isin(rank_jumpers)
].sort_values(
    ['_cap_order', 'Fund Score'], ascending=[True, False]
).reset_index(drop=True)

# Near breakout within 10%
consol_near = tech_df[
    (tech_df['In Consolidation'] == True) &
    (tech_df['Pct to Breakout'] >= -10) &
    (tech_df['Pct to Breakout'] <= 0)
].sort_values('Pct to Breakout', ascending=False).reset_index(drop=True)

# Breakout confirmed (from tracker — above range)
breakout_confirmed = tech_df[
    tech_df['Weeks_Count'].notna() &
    (tech_df['Pct to Breakout'] > 0)
].copy()
if len(breakout_confirmed) > 0:
    breakout_confirmed['_cap_order'] = breakout_confirmed['Cap Category'].apply(cap_order)
    breakout_confirmed = breakout_confirmed.sort_values(
        ['_cap_order', 'Pct to Breakout'], ascending=[True, False]
    )

# Reversal candidates — top 10 per cap by Reversal Score
all_reversal    = tech_df[tech_df['Best Setup'] == 'Reversal'].copy()
reversal_by_cap = {}
for cap in CAP_ORDER:
    subset = all_reversal[all_reversal['Cap Category'] == cap].copy()
    reversal_by_cap[cap] = subset.sort_values(
        'Reversal Score', ascending=False
    ).head(10)

print(f"Data slices ready:")
print(f"  Tier1 Momentum   : {len(tier1_mom)}")
print(f"  Tier2A Watchlist : {len(tier2a)}")
print(f"  Tier2B Base      : {len(tier2b)}")
print(f"  Tier3 Momentum   : {len(tier3_mom)}")
print(f"  Near Breakout    : {len(consol_near)}")
print(f"  Breakout Conf    : {len(breakout_confirmed)}")
print(f"  Reversal stocks  : {len(all_reversal)}")

# ── REPORT GENERATOR (exact Day 8) ───────────────────────────
def generate_report(is_short):
    lines  = []
    serial = 1

    def p(line=''):
        lines.append(str(line))

    mode_label = 'SHORT' if is_short else 'LONG'
    today      = datetime.now().strftime('%d %B %Y')

    p(f"{'='*78}")
    p(f"  AI STOCK SCREENER — WEEKLY REPORT ({mode_label})")
    p(f"  {today}  |  Universe: {len(tech_df)} stocks")
    p(f"{'='*78}")
    p(f"  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + cap")
    p(f"  Week Vol = this week avg vs prior 15-day avg")
    p(f"  Break Vol= this week avg vs consolidation period avg")
    p(f"  ↑ = jumped 10%+ in rank vs last week")

    # ── SECTOR SUMMARY (both short and long) ─────────────────
    p(f"\n{'─'*78}")
    p(f"  SECTOR TREND SUMMARY")
    p(f"{'─'*78}")
    p(f"  {'Sector':42} {'Trend':28} Detail")
    p(f"  {'─'*76}")
    for sector in sorted(SECTOR_INDEX_MAP.keys()):
        label, detail = get_sector_trend(sector)
        p(f"  {sector:42} {label:28} {detail}")

    # ── TIER 1 MOMENTUM ───────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 1 — MOMENTUM  ({len(tier1_mom)} stocks)")
    p(f"  Fund ≥60 + Tech ≥65 + Momentum setup confirmed")
    p(f"{'─'*78}")

    current_cap = None
    for _, row in tier1_mom.iterrows():
        cap = row['Cap Category']
        if cap != current_cap:
            current_cap = cap
            cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                           'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
            p(f"\n  [{cap_short}] {cap}")
            p(f"  {'─'*68}")
        consol_line = ''
        if row['In Consolidation'] and int(row['Consol Days']) > 0:
            bvol_inf    = get_breakout_vol_inference(
                row['Vol 5D Ratio'], row['Breakout Vol'])
            consol_line = (
                f"\n    Base    : {int(row['Consol Days'])}d "
                f"({row['Consol Label']}) | "
                f"{row['Pct to Breakout']:+.1f}% to breakout"
                f"\n    Break   : {bvol_inf}"
            )
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        p(f"""
  #{serial}  {row['Symbol']}{jump_tag}  ({row['Sector']})  MCap {mcap_str(row['Market Cap Cr'])}  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
        serial += 1

    # ── TIER 2A WATCHLIST ─────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 2A — WATCHLIST  (Top 25 | Cap/Sec Rank ≥7 or ↑)")
    p(f"  Setup forming — wait for confirmation before entering")
    p(f"{'─'*78}")

    for _, row in tier2a.iterrows():
        consol_tag = f"  [{int(row['Consol Days'])}d base]" \
                     if row['In Consolidation'] else ''
        jump_tag   = ' ↑' if is_rank_jumper(row) else ''
        p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
          f"Rs{row['Current Price']:.2f}  "
          f"MCap {mcap_str(row['Market Cap Cr'])}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
        p(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
        p(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x "
          f"({row['Vol Label']}) → {row['Vol Inference']}")
        p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
        serial += 1

    # ── TIER 2B BASE BUILDING ─────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 2B — BASE BUILDING  (Top 25 | Cap/Sec Rank ≥7 or ↑)")
    p(f"  Long consolidation base — watch for volume breakout")
    p(f"{'─'*78}")

    for _, row in tier2b.iterrows():
        bvol_inf = get_breakout_vol_inference(
            row['Vol 5D Ratio'], row['Breakout Vol'])
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
          f"Rs{row['Current Price']:.2f}  "
          f"MCap {mcap_str(row['Market Cap Cr'])}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{int(row['Consol Days'])}d  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
        p(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
        p(f"      Volume    : {bvol_inf}")
        p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
        serial += 1

    # ── TIER 3 MOMENTUM (long only) ───────────────────────────
    if not is_short:
        p(f"\n{'─'*78}")
        p(f"  TIER 3 — MOMENTUM WAITING  "
          f"({len(tier3_mom)} shown | Cap/Sec Rank ≥7 or ↑)")
        p(f"  Good businesses with momentum — setup not yet confirmed")
        p(f"{'─'*78}")

        current_cap = None
        for _, row in tier3_mom.iterrows():
            cap = row['Cap Category']
            if cap != current_cap:
                current_cap = cap
                cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                               'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
                p(f"\n  [{cap_short}] {cap}")
                p(f"  {'─'*68}")
            jump_tag = ' ↑' if is_rank_jumper(row) else ''
            if row['In Consolidation'] and row['Breakout Vol'] > 0:
                vol_line = get_breakout_vol_inference(
                    row['Vol 5D Ratio'], row['Breakout Vol'])
            else:
                vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x "
                            f"({row['Vol Label']}) → {row['Vol Inference']}")
            p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
              f"Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Sec:{row['Sector Score']:4.1f}  "
              f"Cap:{row['Cap Score']:4.1f}")
            p(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
              f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
            p(f"      Volume    : {vol_line}")
            p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
            serial += 1

    # ── BREAKOUT CONFIRMED ────────────────────────────────────
    if len(breakout_confirmed) > 0:
        p(f"\n{'─'*78}")
        p(f"  🚀 BREAKOUT CONFIRMED  ({len(breakout_confirmed)} stocks)")
        p(f"  Stocks that broke above consolidation range")
        p(f"  Shown for 3 weeks from breakout date")
        p(f"{'─'*78}")
        p(f"  {'Symbol':12} {'Price':>9} {'MCap':>12} {'Above%':>7} "
          f"{'Weeks':>6} {'WkVol':>7} {'RSI':>5}  Breakout Date")
        p(f"  {'─'*85}")

        current_cap = None
        for _, row in breakout_confirmed.iterrows():
            cap = row['Cap Category']
            if cap != current_cap:
                current_cap = cap
                cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                               'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
                p(f"\n  [{cap_short}] {cap}")
            weeks    = int(row['Weeks_Count']) if pd.notna(row['Weeks_Count']) else 1
            bk_date  = row['First_Breakout_Date'] \
                       if pd.notna(row['First_Breakout_Date']) else today_str
            pct      = row['Pct to Breakout']
            wvol     = row['Vol 5D Ratio']
            wvol_tag = '⚡' if wvol >= 1.5 else ' '
            p(f"  {row['Symbol']:12} "
              f"Rs{row['Current Price']:>8.2f} "
              f"{mcap_str(row['Market Cap Cr']):>12} "
              f"{pct:>+6.1f}%  "
              f"Wk{weeks}/3  "
              f"{wvol:.2f}x{wvol_tag}  "
              f"RSI{row['RSI']:>3.0f}  "
              f"[{bk_date}]")

    # ── NEAR BREAKOUT ─────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  NEAR BREAKOUT BASE WATCH  "
      f"({len(consol_near)} stocks within 10% of breakout)")
    p(f"  Break Vol > 1.5x = volume building — high priority watch")
    p(f"{'─'*78}")
    p(f"  {'Symbol':12} {'Price':>9} {'MCap':>12} {'Days':>5} "
      f"{'ToBreak':>8} {'WkVol':>7} {'BrkVol':>7}  Status")
    p(f"  {'─'*85}")

    for _, row in consol_near.iterrows():
        pct      = row['Pct to Breakout']
        bvol     = row['Breakout Vol']
        wvol     = row['Vol 5D Ratio']
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        status   = 'NEAR BREAKOUT ⚡' if pct > -5 else 'Approaching'
        bvol_tag = '⚡' if bvol >= 1.5 else '~' if bvol >= 0.8 else ' '
        p(f"  {(row['Symbol'] + jump_tag):12} "
          f"Rs{row['Current Price']:>8.2f} "
          f"{mcap_str(row['Market Cap Cr']):>12} "
          f"{int(row['Consol Days']):>5}d "
          f"{pct:>+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x{bvol_tag}  "
          f"[{status}]")

    # ── REVERSAL CANDIDATES ───────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  BULLISH REVERSAL CANDIDATES  (Top 10 per cap category)")
    p(f"  Oversold + divergence forming — wait for confirmation")
    p(f"{'─'*78}")

    for cap in CAP_ORDER:
        subset = reversal_by_cap.get(cap, pd.DataFrame())
        if len(subset) == 0:
            continue
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
        p(f"\n  [{cap_short}] {cap}")
        p(f"  {'─'*68}")
        for _, row in subset.iterrows():
            jump_tag   = ' ↑' if is_rank_jumper(row) else ''
            tier_short = (row['Tier']
                          .replace('TIER 1 — BUY NOW (Reversal)', 'T1-Rev')
                          .replace('TIER 1 — BUY NOW (Momentum)', 'T1-Mom')
                          .replace('TIER 1 — BREAKOUT IMMINENT',  'T1-Brkout')
                          .replace('TIER 2 — WATCHLIST',          'T2')
                          .replace('TIER 2 — BASE BUILDING',      'T2B')
                          .replace('TIER 3 — WAITING',            'T3'))
            p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
              f"Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"[{tier_short}]  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Rev:{row['Reversal Score']:.0f}  "
              f"Sec:{row['Sector Score']:4.1f}  "
              f"Cap:{row['Cap Score']:4.1f}")
            p(f"      Tech    : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
              f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
            p(f"      Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x "
              f"({row['Vol Label']}) → {row['Vol Inference']}")
            p(f"      Sector  : {row['Sector Trend']} | {row['Sector Detail']}")
            serial += 1

    # ── RANK JUMPERS ──────────────────────────────────────────
    if rank_jumpers:
        p(f"\n{'─'*78}")
        p(f"  ↑ RANK JUMPERS THIS WEEK  ({len(rank_jumpers)} stocks)")
        p(f"{'─'*78}")
        jumper_df = tech_df[tech_df['Symbol'].isin(rank_jumpers)].sort_values(
            'Fund Score', ascending=False)
        for _, row in jumper_df.iterrows():
            p(f"  {row['Symbol']:12}  Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Sec:{row['Sector Score']:4.1f}  "
              f"Cap:{row['Cap Score']:4.1f}  "
              f"{row['Tier']}  {row['Sector']}")

    # ── FOOTER ────────────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
    p(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
    p(f"  Week Vol = this week avg vs prior 15-day avg")
    p(f"  Break Vol= this week avg vs consolidation period avg")
    p(f"  ↑ = jumped 10%+ in Cap or Sector rank vs last week")
    p(f"{'─'*78}")

    return lines

# ── GENERATE + SAVE BOTH REPORTS ─────────────────────────────
today_file   = datetime.now().strftime('%Y%m%d')

short_lines  = generate_report(is_short=True)
long_lines   = generate_report(is_short=False)

short_path   = os.path.join(REPORTS_DIR, f'weekly_report_short_{today_file}.txt')
long_path    = os.path.join(REPORTS_DIR, f'weekly_report_long_{today_file}.txt')

with open(short_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(short_lines))
with open(long_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(long_lines))

# Print short report to screen
print('\n'.join(short_lines))

print(f"\n✅ Cell 10B — Weekly technical reports saved")
print(f"   Short : {short_path}")
print(f"   Long  : {long_path}")

Data slices ready:
  Tier1 Momentum   : 88
  Tier2A Watchlist : 25
  Tier2B Base      : 25
  Tier3 Momentum   : 54
  Near Breakout    : 35
  Breakout Conf    : 10
  Reversal stocks  : 20
  AI STOCK SCREENER — WEEKLY REPORT (SHORT)
  13 April 2026  |  Universe: 752 stocks
  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg
  ↑ = jumped 10%+ in rank vs last week

──────────────────────────────────────────────────────────────────────────────
  SECTOR TREND SUMMARY
──────────────────────────────────────────────────────────────────────────────
  Sector                                     Trend                        Detail
  ────────────────────────────────────────────────────────────────────────────
  Agriculture                                Strong Uptrend ↑↑            RSI 67.4 | ADX 27.9 | MACD ▲ | EMA ↓↓↓ | Above EMA200
  Automobile and Auto Components            

In [23]:
# ── CELL 10: GENERATE ML REPORT ───────────────────────────────

tech_df = pd.read_csv(os.path.join(DATA_DIR, 'technical_report_full.csv'))

# ── CONFIDENCE THRESHOLDS ─────────────────────────────────────
CONF_THRESHOLD = {
    'Bullish Continual' : 40.0,
    'Bullish'           : 50.0,
    'Reversal'          : 50.0,
}

# ── TECH TREND ────────────────────────────────────────────────
def get_tech_trend(row):
    try:
        price  = float(row['Current Price'])
        ema20  = float(row['EMA20'])
        ema50  = float(row['EMA50'])
        ema200 = float(row['EMA200'])
        adx    = float(row['ADX'])
        rsi    = float(row['RSI'])
        macd   = float(row['MACD Hist'])
        vol    = float(row['Vol Ratio'])

        full_bull_align = price > ema20 > ema50 > ema200
        partial_bull    = price > ema50 > ema200
        full_bear_align = price < ema20 and ema20 < ema50 and ema50 < ema200
        partial_bear    = price < ema200 and ema20 < ema50

        strong_trend  = adx > 25
        mild_trend    = adx > 20
        bull_momentum = rsi > 55 and macd > 0
        bear_momentum = rsi < 45 and macd < 0
        high_vol      = vol > 1.0

        if full_bull_align and strong_trend and bull_momentum and high_vol:
            return 'Str-Up'
        elif partial_bull and (mild_trend or rsi > 50 or macd > 0):
            return 'Up'
        elif full_bear_align and strong_trend and bear_momentum and high_vol:
            return 'Str-Dn'
        elif partial_bear and (mild_trend or macd < 0):
            return 'Down'
        else:
            return 'Side'
    except:
        return '?'

tech_df['Tech_Trend'] = tech_df.apply(get_tech_trend, axis=1)

# ── HELPER FUNCTIONS ──────────────────────────────────────────
def mcap_str(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 100000: return f"Rs{v/100000:.1f}L Cr"
        return f"Rs{v:,.0f}Cr"
    except:
        return "Rs—"

def tier_short(tier):
    return (str(tier)
            .replace('TIER 1 — ', 'T1-')
            .replace('TIER 2 — ', 'T2-')
            .replace('TIER 3 — ', 'T3-')
            .replace('BUY NOW (Momentum)', 'MOM')
            .replace('BUY NOW (Reversal)', 'REV')
            .replace('BREAKOUT IMMINENT',  'BRKOUT')
            .replace('WATCHLIST',          'WATCH')
            .replace('BASE BUILDING',      'BASE')
            .replace('WAITING',            'WAIT'))

# ── FILTER BUY-SIDE STOCKS ────────────────────────────────────
buy_labels = ['Bullish Continual', 'Bullish', 'Reversal']

filtered = tech_df[
    tech_df['ML_Prediction'].isin(buy_labels) &
    tech_df['ML_Confidence'].notna()
].copy()

# Apply confidence thresholds
filtered = filtered[
    filtered.apply(
        lambda r: r['ML_Confidence'] >= CONF_THRESHOLD.get(r['ML_Prediction'], 50),
        axis=1
    )
].copy()

filtered = filtered.sort_values('ML_Confidence', ascending=False).reset_index(drop=True)

print(f"Stocks passing confidence threshold:")
print(filtered['ML_Prediction'].value_counts().to_string())
print(f"Total : {len(filtered)}")

# ── REPORT BUILDER ────────────────────────────────────────────
def build_report(df, fmt='long'):
    now       = datetime.now().strftime('%Y-%m-%d %H:%M')
    lines     = []
    separator = "=" * 74

    lines.append(separator)
    lines.append(f"  AI STOCK SCREENER — ML REPORT ({fmt.upper()})")
    lines.append(f"  Generated : {now}")
    lines.append(f"  Universe  : 752 stocks | Buy signals: {len(df)}")
    lines.append(separator)

    for label in buy_labels:
        section = df[df['ML_Prediction'] == label].copy()
        if len(section) == 0:
            continue

        lines.append(f"\n{'─' * 74}")
        lines.append(f"  {label.upper()}  ({len(section)} stocks)")
        if label == 'Bullish Continual':
            lines.append(f"  Already in uptrend — ML confirms continuation  [threshold: ≥40%]")
        elif label == 'Bullish':
            lines.append(f"  No clear setup yet — ML sees upside potential  [threshold: ≥50%]")
        elif label == 'Reversal':
            lines.append(f"  In downtrend — ML sees high bounce probability [threshold: ≥50%]")
        lines.append(f"{'─' * 74}")

        if fmt == 'short':
            # Header
            lines.append(f"  {'#':>3}  {'Symbol':<14} {'Cap':>12}  {'SecRnk':>6}  "
                         f"{'CapRnk':>6}  {'Conf':>5}  {'25d%':>5}  {'45d%':>5}  "
                         f"{'Trend':<6}  {'Tier'}")
            lines.append(f"  {'─'*3}  {'─'*14} {'─'*12}  {'─'*6}  "
                         f"{'─'*6}  {'─'*5}  {'─'*5}  {'─'*5}  "
                         f"{'─'*6}  {'─'*10}")

            for i, (_, row) in enumerate(section.iterrows(), 1):
                lines.append(
                    f"  {i:>3}  {row['Symbol']:<14} "
                    f"{mcap_str(row['Market Cap Cr']):>12}  "
                    f"{row['Sector Score']:>6.1f}  "
                    f"{row['Cap Score']:>6.1f}  "
                    f"{row['ML_Confidence']:>5.1f}  "
                    f"{row['Forecast_25d_Pct']:>+5.1f}  "
                    f"{row['Forecast_45d_Pct']:>+5.1f}  "
                    f"{row['Tech_Trend']:<6}  "
                    f"{tier_short(row['Tier'])}"
                )

        else:  # long
            for i, (_, row) in enumerate(section.iterrows(), 1):
                lines.append(
                    f"\n  {i}. {row['Symbol']}  |  {row['Sector']}  |  "
                    f"{row['Cap Category']}  |  {mcap_str(row['Market Cap Cr'])}"
                )
                lines.append(
                    f"     Confidence : {row['ML_Confidence']:.1f}%  |  "
                    f"Trend: {row['Tech_Trend']}  |  "
                    f"Tier: {tier_short(row['Tier'])}"
                )
                lines.append(
                    f"     Forecast   : 25d={row['Forecast_25d_Pct']:+.1f}%  "
                    f"45d={row['Forecast_45d_Pct']:+.1f}%  "
                    f"180d={row['Forecast_180d_Pct']:+.1f}%"
                )
                lines.append(
                    f"     Prices     : Now=Rs{row['Current Price']:.0f}  "
                    f"25d=Rs{row['Forecast_25d_Price']:.0f}  "
                    f"45d=Rs{row['Forecast_45d_Price']:.0f}  "
                    f"180d=Rs{row['Forecast_180d_Price']:.0f}"
                )
                lines.append(
                    f"     Scores     : SecRnk={row['Sector Score']:.1f}  "
                    f"CapRnk={row['Cap Score']:.1f}  "
                    f"FundScore={row.get('Fund Score', '—')}"
                )
                lines.append(
                    f"     ML Probs   : BullCont={row['Bullish_Cont_Prob']:.1f}%  "
                    f"BotRev={row['Bottom_Rev_Prob']:.1f}%  "
                    f"TopRev={row['Top_Rev_Prob']:.1f}%"
                )

    lines.append(f"\n{'=' * 74}")
    lines.append(f"  END OF REPORT")
    lines.append(f"{'=' * 74}\n")

    return "\n".join(lines)

# ── GENERATE + SAVE BOTH REPORTS ─────────────────────────────
today = datetime.now().strftime('%Y%m%d')

short_report = build_report(filtered, fmt='short')
long_report  = build_report(filtered, fmt='long')

short_path = os.path.join(REPORTS_DIR, f'ml_report_short_{today}.txt')
long_path  = os.path.join(REPORTS_DIR, f'ml_report_long_{today}.txt')

with open(short_path, 'w', encoding='utf-8') as f:
    f.write(short_report)

with open(long_path, 'w', encoding='utf-8') as f:
    f.write(long_report)

# Print short report to screen
print("\n")
print(short_report)

print(f"✅ Cell 10 — Reports saved")
print(f"   Short : {short_path}")
print(f"   Long  : {long_path}")

Stocks passing confidence threshold:
ML_Prediction
Bullish Continual    59
Bullish              40
Reversal             30
Total : 129


  AI STOCK SCREENER — ML REPORT (SHORT)
  Generated : 2026-04-13 00:13
  Universe  : 752 stocks | Buy signals: 129

──────────────────────────────────────────────────────────────────────────
  BULLISH CONTINUAL  (59 stocks)
  Already in uptrend — ML confirms continuation  [threshold: ≥40%]
──────────────────────────────────────────────────────────────────────────
    #  Symbol                  Cap  SecRnk  CapRnk   Conf   25d%   45d%  Trend   Tier
  ───  ────────────── ────────────  ──────  ──────  ─────  ─────  ─────  ──────  ──────────
    1  GLOBAL              Rs505Cr     8.9    10.0   81.9   +6.3   +9.5  Up      T1-MOM
    2  ORICONENT         Rs1,066Cr     5.7     5.7   81.6   +5.0   +7.6  Up      T3-WAIT
    3  GMDCLTD          Rs18,310Cr     7.5     9.1   81.6   +4.6  +10.7  Up      T1-MOM
    4  AGIIL             Rs3,892Cr    10.0    10.0   8

In [24]:
# ── MIGRATE FILES TO NEW FOLDER STRUCTURE ────────────────────
import shutil

print("Migrating files to new folder structure...")
print("-" * 60)

# ── DEFINE NEW FOLDER STRUCTURE ──────────────────────────────
folders = {
    'universe'    : os.path.join(DATA_DIR, 'universe'),
    'fundamentals': os.path.join(DATA_DIR, 'fundamentals'),
    'prices'      : os.path.join(DATA_DIR, 'prices'),
    'scores'      : os.path.join(DATA_DIR, 'scores'),
    'temp'        : os.path.join(DATA_DIR, 'temp'),
    'reports_tech': os.path.join(DATA_DIR, 'reports', 'technical'),
    'reports_ml'  : os.path.join(DATA_DIR, 'reports', 'ml'),
}

# Create all folders
for name, path in folders.items():
    os.makedirs(path, exist_ok=True)
    print(f"  Created : {path}")

# ── FILE MIGRATION MAP ────────────────────────────────────────
# (source_filename, destination_folder_key)
migration_map = [
    # Universe files
    ('master_stocks.csv',             'universe'),
    ('bse_stocks.csv',                'universe'),
    ('prefilt_passed.csv',            'universe'),
    ('prefilt_results.csv',           'universe'),
    ('quality_passed.csv',            'universe'),
    ('quality_filter_results.csv',    'universe'),

    # Fundamental files
    ('fundamental_metrics_full.csv',  'fundamentals'),
    ('fundamental_scores_full.csv',   'fundamentals'),
    ('raw_stock_data_full.pkl',       'fundamentals'),

    # Price + indicator files
    ('price_data_full.pkl',           'prices'),
    ('indicator_data_full.pkl',       'prices'),

    # Score files
    ('technical_report_full.csv',     'scores'),
    ('ml_scores_full.csv',            'scores'),
    ('last_week_scores.csv',          'scores'),
    ('breakout_tracker.csv',          'scores'),

    # Temp / checkpoint files
    ('tech_checkpoint.pkl',           'temp'),
    ('prefilt_checkpoint.pkl',        'temp'),
    ('fund_scrape_checkpoint.pkl',    'temp'),
]

print(f"\nMigrating files...")
migrated = []
skipped  = []
missing  = []

for filename, folder_key in migration_map:
    src  = os.path.join(DATA_DIR, filename)
    dst  = os.path.join(folders[folder_key], filename)

    if os.path.exists(src):
        shutil.move(src, dst)
        migrated.append(f"  ✅ {filename:45} → {folder_key}/")
    elif os.path.exists(dst):
        skipped.append(f"  ⚠️  {filename:45} already in {folder_key}/ (skipped)")
    else:
        missing.append(f"  ❌ {filename:45} not found")

for m in migrated: print(m)
for s in skipped:  print(s)
for m in missing:  print(m)

# ── MIGRATE WEEKLY REPORTS ────────────────────────────────────
print(f"\nMigrating weekly reports...")
old_reports_dir = os.path.join(DATA_DIR, 'weekly_reports')

if os.path.exists(old_reports_dir):
    for fname in os.listdir(old_reports_dir):
        src = os.path.join(old_reports_dir, fname)
        if fname.startswith('weekly_report_'):
            dst = os.path.join(folders['reports_tech'], fname)
        elif fname.startswith('ml_report_'):
            dst = os.path.join(folders['reports_ml'], fname)
        else:
            dst = os.path.join(folders['reports_tech'], fname)

        if os.path.isfile(src):
            shutil.move(src, dst)
            print(f"  ✅ {fname} → reports/{('ml' if fname.startswith('ml_') else 'technical')}/")

    # Remove old empty folder
    try:
        os.rmdir(old_reports_dir)
        print(f"  🗑️  Removed old weekly_reports/ folder")
    except:
        print(f"  ⚠️  weekly_reports/ not empty — check manually")
else:
    print(f"  ⚠️  weekly_reports/ folder not found — already migrated")

# ── VERIFY FINAL STRUCTURE ────────────────────────────────────
print(f"\nFinal folder structure:")
for root, dirs, files in os.walk(DATA_DIR):
    # Skip hidden folders
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level   = root.replace(DATA_DIR, '').count(os.sep)
    indent  = '  ' * level
    folder  = os.path.basename(root)
    print(f"{indent}📁 {folder}/")
    subindent = '  ' * (level + 1)
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        if size >= 1024*1024:
            size_str = f"{size/1024/1024:.1f}MB"
        elif size >= 1024:
            size_str = f"{size/1024:.1f}KB"
        else:
            size_str = f"{size}B"
        print(f"{subindent}📄 {f} ({size_str})")

print(f"\n✅ Migration complete")

Migrating files to new folder structure...
------------------------------------------------------------
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\universe
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\fundamentals
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\prices
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\scores
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\temp
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\technical
  Created : E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\ml

Migrating files...
  ✅ master_stocks.csv                             → universe/
  ✅ bse_stocks.csv                                → universe/
  ✅ prefilt_passed.csv                            → universe/
  ✅ prefilt_results.csv                           → universe/
  ✅ quality_passed.csv                            → universe/
  ✅ quality_filter_results.csv                    → uni